# Логистическая регрессия для прогноза сделок

Оцениваем модель на одном отложенном тесте. Для итоговой выгрузки обучаем модель на всех объявлениях с доступными метками. В тесте и в итоговом CSV присваиваем `0` объявлениям из файлов аномалий.

## 1. Загрузить данные

По умолчанию используем `features.csv`. Чтобы использовать сокращённый набор, поменяйте `FEATURES_PATH` на `best_features.csv`. Метки и признаки сопоставляем по `listing_id`.

In [53]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [54]:
REPO_DIR = Path("/Users/ynelo/Documents/github/neuroslop_kygarikovs_hakaton")
FEATURES_PATH = REPO_DIR / "data_temp" / "features.csv"
VERIFIED_PATH = REPO_DIR / "data" / "verified.csv"
ANOMALY_FILES = [
    REPO_DIR / "data_temp/listing_ids_of_bad_time.csv",
    REPO_DIR / "data_temp/listing_id_of_high.csv",
    REPO_DIR / "data_temp/listing_id_of_block.csv",
    REPO_DIR / "data_temp/low_listing_ids.csv",
]
OUTPUT_PATH = Path(f"submission_logreg_without_cv_{FEATURES_PATH.stem}.csv")

In [55]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")

feature_columns = features.columns.difference(["seller_id", "is_deal"], sort=False)
X = features[feature_columns]
y = verified.loc[verified.index.isin(X.index), "is_deal"]
X_train = X.loc[y.index]

print(f"Объявлений в матрице: {len(X)}")
print(f"Размеченных объявлений с признаками: {len(y)} из {len(verified)}")

Объявлений в матрице: 15051
Размеченных объявлений с признаками: 294 из 300


## 2. Создать модель

В числовом блоке заменяем бесконечности пропусками, заполняем пропуски медианой и стандартизируем значения. В категориальном блоке заполняем пропуски самым частым значением и применяем one-hot.

Логистическая регрессия использует Elastic Net с `C=0.8` и `l1_ratio=0.5`.

In [56]:
numeric_columns = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_columns = X_train.columns.difference(numeric_columns, sort=False).tolist()


def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns),
])
model = make_pipeline(
    preprocessor,
    LogisticRegression(
    C=0.8,
    l1_ratio=0.5,
    solver="saga",
    max_iter=5000,
    random_state=42,
)
)

## 3. Оценить модель на отложенном тесте

Прогноз можно проверить для объявлений из `verified.csv`, если у них есть признаки или их ID указан в файле аномалий. Делим эти объявления один раз в пропорции 80/20, сохраняя близкую долю сделок в обеих частях. Модель обучается на строках с признаками из обучающей части.

В тестовой части прогноз для аномалий равен `0`, даже если у них нет признаков. Для ROC-AUC их оценка также равна нулю.

In [57]:
anomaly_ids = pd.Index(
    pd.concat(
        [pd.read_csv(path, usecols=["listing_id"])["listing_id"] for path in ANOMALY_FILES],
        ignore_index=True,
    ).dropna().drop_duplicates(),
    name="listing_id",
)


def apply_anomaly_overrides(values, anomaly_ids):
    values = values.reindex(values.index.union(anomaly_ids, sort=False))
    values.loc[anomaly_ids] = 0
    return values

In [58]:
evaluation_y = verified.loc[verified.index.isin(X.index.union(anomaly_ids)), "is_deal"]
train_y, test_y = train_test_split(
    evaluation_y, test_size=0.2, stratify=evaluation_y, random_state=42,
)
train_with_features = train_y.loc[train_y.index.isin(X.index)]
test_with_features = test_y.index.intersection(X.index, sort=False)
test_anomalies = test_y.index.intersection(anomaly_ids, sort=False)

test_model = clone(model).fit(X.loc[train_with_features.index], train_with_features)
test_probability = pd.Series(
    test_model.predict_proba(X.loc[test_with_features])[:, 1],
    index=test_with_features,
)
test_probability = apply_anomaly_overrides(test_probability, test_anomalies).loc[test_y.index]
test_prediction = (test_probability > 0.5).astype(int)

metrics = pd.Series({
    "F1 по сделкам": f1_score(test_y, test_prediction),
    "Точность (precision)": precision_score(test_y, test_prediction),
    "Полнота (recall)": recall_score(test_y, test_prediction),
    "ROC-AUC": roc_auc_score(test_y, test_probability),
    "Доля верных ответов (accuracy)": accuracy_score(test_y, test_prediction),
}, name="Значение")

print(f"Обучение: {len(train_with_features)} размеченных объявлений с признаками")
print(f"Тест: {len(test_y)} объявлений, из них {len(test_anomalies)} аномалий")
metrics.round(4).to_frame()

Обучение: 236 размеченных объявлений с признаками
Тест: 60 объявлений, из них 3 аномалий


,Значение
F1 по сделкам,0.7660
Точность (precision),0.7500
Полнота (recall),0.7826
ROC-AUC,0.8472
Доля верных ответов (accuracy),0.8167


## 4. Обучить итоговую модель и получить прогнозы

Обучаем модель на всех объявлениях с доступными метками. Для каждой строки выбранного CSV получаем прогноз `is_deal` равный `0` или `1`.

In [59]:
model.fit(X_train, y)
predictions = X.index.to_frame(index=False).assign(is_deal=model.predict(X))

## 5. Проставить нули для аномалий и сохранить CSV

По `listing_id` присваиваем `is_deal=0` строкам из файлов аномалий. Если объявления ещё нет в прогнозе, добавляем строку с нулём. Чтобы обработать остальные файлы, перечислите их в `ANOMALY_FILES`. Каждый файл должен содержать столбец `listing_id`.

In [60]:
labels = predictions.set_index("listing_id")["is_deal"]
submission = apply_anomaly_overrides(labels, anomaly_ids).astype(int).reset_index(name="is_deal")
submission.to_csv(OUTPUT_PATH, index=False)

print(f"Метка 0 проставлена для {len(anomaly_ids)} ID из файлов аномалий")
print(f"Добавлено строк: {len(submission) - len(predictions)}")
print(f"Метки сохранены для {len(submission)} объявлений в {OUTPUT_PATH.resolve()}")
submission.head()

Метка 0 проставлена для 474 ID из файлов аномалий
Добавлено строк: 223
Метки сохранены для 15274 объявлений в /Users/ynelo/Documents/github/neuroslop_kygarikovs_hakaton/submission_logreg_without_cv_features.csv


,listing_id,is_deal
0,145079,1
1,100057,0
2,102500,0
3,157287,1
4,110242,0
